In [14]:
import numpy as np
import sionna.rt as rt

practice_scene = rt.load_scene()

In [15]:
tx = rt.Transmitter(
    name = "test_tx",
    position = [0, 0, 2]
)

practice_scene.add(tx)

In [16]:
rx_a = rt.Receiver(
    name = "rx_a",
    position = [5, 0, 1]
)

rx_b = rt.Receiver(
    name = "rx_b",
    position = [10, 0, 1]
)

practice_scene.add(rx_a)
practice_scene.add(rx_b)

rec_a = practice_scene.get("rx_a")
rec_a.position = [7, 0, 1]

rec_b = practice_scene.get("rx_b")
practice_scene.remove(rec_b.name)

print(rx_a.position)
print(rx_a.name)
print(practice_scene.get("rx_b"))

rx_b.position = [12, 0, 1]

practice_scene.add(rx_b)

print(practice_scene.receivers.keys())
print(practice_scene.get("rx_b").position)

[[7, 0, 1]]
rx_a
None
dict_keys(['rx_a', 'rx_b'])
[[12, 0, 1]]


In [17]:
practice_scene.frequency = 3.5e9

practice_scene.tx_array = rt.PlanarArray(
    num_rows = 1,
    num_cols = 1,
    vertical_spacing = 0.5,
    horizontal_spacing = 0.5,
    pattern = "iso",
    polarization = "V"
)

practice_scene.rx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization= "V"
)

practice_scene.all_set(radio_map=False)

In [18]:
practice_solver = rt.PathSolver()

practice_paths = practice_solver(
    scene = practice_scene,
    max_depth=0
)

print(list(practice_scene.receivers.keys()))
print(practice_paths.valid)
delays = np.array(practice_paths.tau)
print(delays.shape)

delays_ns = delays * 1e+9
print(delays_ns[0, 0, 0])
print(delays_ns[1, 0, 0])

['rx_a', 'rx_b']
[[[True]],
 [[True]]]
(2, 1, 1)
23.586542
40.166435


In [20]:
tx_position = np.array(practice_scene.get("test_tx").position)
rx_position = np.array(practice_scene.get("rx_a").position)

distance = np.linalg.norm(rx_position - tx_position)

delay_for_first_receiver_ns = (distance / 299_792_458) * 1e+9
print("My value:", delay_for_first_receiver_ns, ", value calculated by sionna:", delays_ns[0, 0, 0])

rx_b_position = np.array(practice_scene.get("rx_b").position)

distance_second = np.linalg.norm(rx_b_position - tx_position)

delay_for_second_receiver_ns = (distance_second / 299_792_458) * 1e+9
print("My second value:", delay_for_second_receiver_ns, ", value calculated by sionna:", delays_ns[1, 0, 0])



My value: 23.586544 , value calculated by sionna: 23.586542
My second value: 40.16644 , value calculated by sionna: 40.166435


In [24]:
first_matches = np.isclose(
    delays_ns[0, 0, 0],
    delay_for_first_receiver_ns,
    rtol = 1e-6,
    atol = 0.0
)

print(first_matches)

second_matches = np.isclose(
    delays_ns[1, 0, 0],
    delay_for_second_receiver_ns,
    rtol = 1e-6,
    atol = 0.0
)

print(second_matches)

false_second_matches = np.isclose(
    delays_ns[1, 0, 0] + 1,
    delay_for_second_receiver_ns,
    rtol = 1e-6,
    atol = 0.0
)

print(false_second_matches)

True
True
False
